In [12]:
# Install correct stable versions
!pip uninstall -y langchain langchain-community langchain-core
!pip install langchain==0.1.17 langchain-community faiss-cpu transformers sentence-transformers pypdf

Found existing installation: langchain 0.1.17
Uninstalling langchain-0.1.17:
  Successfully uninstalled langchain-0.1.17
Found existing installation: langchain-community 0.0.38
Uninstalling langchain-community-0.0.38:
  Successfully uninstalled langchain-community-0.0.38
Found existing installation: langchain-core 0.1.53
Uninstalling langchain-core-0.1.53:
  Successfully uninstalled langchain-core-0.1.53
  Using cached langchain-0.1.17-py3-none-any.whl.metadata (13 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.0.38-py3-none-any.whl.metadata (8.7 kB)
  Using cached langchain_core-0.1.53-py3-none-any.whl.metadata (5.9 kB)
Using cached langchain-0.1.17-py3-none-any.whl (867 kB)
Using cached langchain_community-0.0.38-py3-none-any.whl (2.0 MB)
Using cached langchain_core-0.1.53-py3-none-any.whl (303 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavi

In [1]:
# Upload multiple PDFs
from google.colab import files
uploaded = files.upload()

Saving Artificial_intelligence_in_India - Copy.pdf to Artificial_intelligence_in_India - Copy.pdf
Saving Artificial_intelligence_in_India.pdf to Artificial_intelligence_in_India.pdf
Saving Climate_change_in_India.pdf to Climate_change_in_India.pdf
Saving COVID-19_pandemic.pdf to COVID-19_pandemic.pdf
Saving ISRO.pdf to ISRO.pdf
Saving Temples_of_modern_India.pdf to Temples_of_modern_India.pdf


In [2]:
# Create data folder and move PDFs into it
import os, shutil

os.makedirs("data", exist_ok=True)

for file in uploaded:
    shutil.move(file, "data/" + file)

print("✅ PDFs uploaded:", os.listdir("data"))

✅ PDFs uploaded: ['Artificial_intelligence_in_India.pdf', 'Climate_change_in_India.pdf', 'Temples_of_modern_India.pdf', 'ISRO.pdf', 'Artificial_intelligence_in_India - Copy.pdf', 'COVID-19_pandemic.pdf']


In [3]:
# Load all PDF documents
from langchain_community.document_loaders import PyPDFLoader

documents = []

for file in os.listdir("data"):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(f"data/{file}")
        documents.extend(loader.load())

print("✅ Documents loaded:", len(documents))

✅ Documents loaded: 282


In [4]:
# Split documents into chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,   # bigger chunks = faster
    chunk_overlap=100  # keeps context
)

chunks = splitter.split_documents(documents)

print("✅ Total chunks:", len(chunks))

✅ Total chunks: 1193


In [6]:
# Convert text into vector embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("✅ Embeddings ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embeddings ready


In [7]:
# Store embeddings in FAISS vector DB
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(chunks, embeddings)

# Save locally
db.save_local("faiss_index")

print("✅ Database created and saved")

✅ Database created and saved


In [8]:
# Create retrieval + generation pipeline
from langchain.chains import RetrievalQA
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

# Retriever (search top 3 chunks)
retriever = db.as_retriever(search_kwargs={"k": 3})

# Lightweight free model
pipe = pipeline("text-generation", model="gpt2")

llm = HuggingFacePipeline(pipeline=pipe)

# Combine retrieval + LLM
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

print("✅ Chatbot ready")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Chatbot ready


In [10]:
while True:
    query = input("Ask (type 'exit' to stop): ")

    if query.lower() == "exit":
        print("Stopped!")
        break

    result = qa(query)

    print("\nAnswer:", result["result"])

    print("\nSources:")
    for doc in result["source_documents"]:
        print(doc.metadata)

Ask (type 'exit' to stop): what is artificial intellgence?


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Pune
IIT Tirupati Navavishkar I-Hub Foundation IIT Tirupati Positioning & Precision
Technologies
IIT Bhilai Innovation and Technology
Foundation IIT Bhilai Fintech
India currently does not have specific laws regulating artificial intelligence (AI). However, the Indian
government has introduced several initiatives and guidelines aimed at the responsible development and
deployment of AI technologies.[87][88] The Indian government has tasked NITI Aayog, its apex public
policy think tank, with establishing guidelines and policies for AI. In 2018, NITI Aayog released the
National Strategy for Artificial Intelligence, also known as #AIForAll, which focuses on healthcare,
agriculture, education, smart cities, and smart mobility.[89]
In 2021, NITI Aayog published the "Principles for Responsible AI," addressing ethical consi

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


IndexError: index out of range in self